In [ ]:
import numpy as np
import casadi as cs
import matplotlib.pyplot as plt

np.random.seed(0)

# Problem parameters
h = 0.01  # 100Hz beacon freq
Q = 0.01  # Small Q, assume constant longitudinal velocity (approx 10m/s)
R = 1.5**2  # Measurement noise variance

# Linear dynamics matrices (arc length and velocity)
A = np.array([[1, h], [0, 1]])
G = np.array([[0], [1]])

# Assume the vehicle stays exactly at the center of the lane
R_c = 48.0 

# Place beacons to avoid ambiguity: 2 outside, 1 inside
beacons = np.array([
    [20, 10],      
    [100, 100],    
    [-100, 100]    
])
ny = len(beacons)
R_mat = R * np.eye(ny)

# Assume initial state is relatively well known (small P0)
x0_tilde = np.array([[0], [10]]) 
P0 = np.array([[1, 0], [0, 1]])  

# CasADi symbolic variables for nonlinear measurement model
x_sym = cs.SX.sym('x', 2)
v_sym = cs.SX.sym('v', ny)

p_t = x_sym[0]
pos_x = R_c * cs.cos(p_t / R_c)
pos_y = R_c * cs.sin(p_t / R_c)

h_list = []
for i in range(ny):
    bx = beacons[i, 0]
    by = beacons[i, 1]
    # Nonlinear distance formula
    dist = cs.sqrt((pos_x - bx)**2 + (pos_y - by)**2)
    h_list.append(dist + v_sym[i])

h_sym = cs.vertcat(*h_list)

# Jacobian for EKF
jhx = cs.Function('jhx', [x_sym, v_sym], [cs.jacobian(h_sym, x_sym)])
h_func = cs.Function('h_func', [x_sym, v_sym], [h_sym])

def measurement_update(sigma_tu, x_tu, y):
    C_mat = np.array(jhx(x_tu, np.zeros(ny))).astype(float)
    Z = C_mat @ sigma_tu @ C_mat.T + R_mat
    y_pred = np.array(h_func(x_tu, np.zeros(ny))).flatten()
    
    x_mu = x_tu + (sigma_tu @ C_mat.T @ np.linalg.solve(Z, y - y_pred)).reshape(-1, 1)
    sigma_mu = sigma_tu - sigma_tu @ C_mat.T @ np.linalg.solve(Z, C_mat @ sigma_tu)
    return sigma_mu, x_mu

def time_update(sigma_mu, x_mu):
    x_tu = A @ x_mu
    sigma_tu = A @ sigma_mu @ A.T + Q * (G @ G.T)
    return sigma_tu, x_tu